In [1]:
import os
from dotenv import load_dotenv
from alpaca.data.requests import StockBarsRequest, StockLatestQuoteRequest
from alpaca.data.timeframe import TimeFrame
from datetime import datetime
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.trading.client import TradingClient
from alpaca.data.live import StockDataStream
from alpaca.data.enums import DataFeed
import threading
from datetime import datetime, timezone

from equity_data_platform.clients import create_historical_client
from equity_data_platform.config import get_settings

In [2]:
# load_dotenv()

# api_key = os.getenv("APCA_API_KEY_ID")
# secret_key = os.getenv("APCA_API_SECRET_KEY")

# if not api_key or not secret_key:
#     raise ValueError("Alpaca API keys were not found in .env")

In [3]:
# client = StockHistoricalDataClient(api_key, secret_key)
settings = get_settings()
client = create_historical_client(settings)

In [4]:
symbol = settings.symbols[0:4]

request = StockBarsRequest(
    symbol_or_symbols=symbol,
    timeframe=TimeFrame.Day,
    start=datetime(2026, 10, 1),
    end=datetime(2026, 10, 16),
    feed=DataFeed(settings.feed),
)

In [5]:
# Retrieve daily bars for Bitcoin in a DataFrame and printing it
response = client.get_stock_bars(request)

# Convert to dataframe
response

{   'data': {   'AAPL': [   {   'close': 330.44,
                                'high': 332.47,
                                'low': 325.81,
                                'open': 329.55,
                                'symbol': 'AAPL',
                                'timestamp': datetime.datetime(2026, 10, 1, 4, 0, tzinfo=TzInfo(0)),
                                'trade_count': 19383.0,
                                'volume': 988015.0,
                                'vwap': 329.36679},
                            {   'close': 333.75,
                                'high': 334.525,
                                'low': 330.66,
                                'open': 333.16,
                                'symbol': 'AAPL',
                                'timestamp': datetime.datetime(2026, 10, 2, 4, 0, tzinfo=TzInfo(0)),
                                'trade_count': 18894.0,
                                'volume': 841636.0,
                                'vwap': 332.9

In [6]:
for symbol, bars in response.data.items():
    print(symbol)
    print("record count:", len(bars))

    if bars:
        print(bars[0].model_dump(mode="json"))

AMZN
record count: 11
{'symbol': 'AMZN', 'timestamp': '2026-10-01T04:00:00Z', 'open': 251.51, 'high': 251.8, 'low': 246.12, 'close': 248.27, 'volume': 1142402.0, 'trade_count': 14531.0, 'vwap': 248.383864}
MSFT
record count: 11
{'symbol': 'MSFT', 'timestamp': '2026-10-01T04:00:00Z', 'open': 519.765, 'high': 522.805, 'low': 512.24, 'close': 512.71, 'volume': 788881.0, 'trade_count': 16533.0, 'vwap': 516.178013}
NVDA
record count: 11
{'symbol': 'NVDA', 'timestamp': '2026-10-01T04:00:00Z', 'open': 229.73, 'high': 232.28, 'low': 228.19, 'close': 231.04, 'volume': 2319590.0, 'trade_count': 28730.0, 'vwap': 230.463041}
AAPL
record count: 11
{'symbol': 'AAPL', 'timestamp': '2026-10-01T04:00:00Z', 'open': 329.55, 'high': 332.47, 'low': 325.81, 'close': 330.44, 'volume': 988015.0, 'trade_count': 19383.0, 'vwap': 329.36679}


In [7]:
df = response.df
df.tail()

open     high     low    close  \
symbol timestamp                                                      
AAPL   2026-10-09 04:00:00+00:00  331.685  338.600  330.72  336.470   
       2026-10-12 04:00:00+00:00  338.130  342.615  337.62  341.045   
       2026-10-13 04:00:00+00:00  344.675  348.110  340.75  341.330   
       2026-10-14 04:00:00+00:00  344.605  347.090  341.72  343.010   
       2026-10-15 04:00:00+00:00  347.025  347.720  341.62  341.910   

                                     volume  trade_count        vwap  
symbol timestamp                                                      
AAPL   2026-10-09 04:00:00+00:00  2132531.0      30595.0  334.173843  
       2026-10-12 04:00:00+00:00   664699.0      16339.0  340.608845  
       2026-10-13 04:00:00+00:00  1307464.0      23639.0  344.299790  
       2026-10-14 04:00:00+00:00  1874329.0      32914.0  344.103129  
       2026-10-15 04:00:00+00:00  1111639.0      24529.0  344.214777

In [8]:
bars_df = df.reset_index()

print("Shape:", bars_df.shape)
print("\nColumns:")
print(bars_df.columns.tolist())

print("\nData types:")
print(bars_df.dtypes)

print("\nMissing values:")
print(bars_df.isna().sum())

print("\nRows per symbol:")
print(bars_df.groupby("symbol").size())

print("\nSample:")
display(bars_df.head())

Shape: (44, 9)

Columns:
['symbol', 'timestamp', 'open', 'high', 'low', 'close', 'volume', 'trade_count', 'vwap']

Data types:
symbol                         str
timestamp      datetime64[us, UTC]
open                       float64
high                       float64
low                        float64
close                      float64
volume                     float64
trade_count                float64
vwap                       float64
dtype: object

Missing values:
symbol         0
timestamp      0
open           0
high           0
low            0
close          0
volume         0
trade_count    0
vwap           0
dtype: int64

Rows per symbol:
symbol
AAPL    11
AMZN    11
MSFT    11
NVDA    11
dtype: int64

Sample:


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap
0,AMZN,2026-10-01 04:00:00+00:00,251.510,251.80,246.12,248.270,1142402.0,14531.0,248.383864
1,AMZN,2026-10-02 04:00:00+00:00,251.635,253.54,250.02,251.420,1021515.0,13799.0,251.401462
2,AMZN,2026-10-05 04:00:00+00:00,250.690,254.49,250.23,251.285,1184678.0,15065.0,252.312298
3,AMZN,2026-10-06 04:00:00+00:00,253.520,256.67,251.09,256.310,1062891.0,15868.0,254.992326
4,AMZN,2026-10-07 04:00:00+00:00,254.265,260.12,253.21,259.900,1095859.0,13437.0,257.970581


In [9]:
required_columns = {
    "symbol",
    "timestamp",
    "open",
    "high",
    "low",
    "close",
    "volume",
    "trade_count",
    "vwap",
}

missing_columns = required_columns - set(bars_df.columns)

if missing_columns:
    raise ValueError(f"Missing columns: {missing_columns}")

checks = {
    "no missing values": not bars_df[list(required_columns)].isna().any().any(),

    "unique symbol/timestamp": not bars_df.duplicated(
        subset=["symbol", "timestamp"]
    ).any(),

    "high >= open": (bars_df["high"] >= bars_df["open"]).all(),

    "high >= close": (bars_df["high"] >= bars_df["close"]).all(),

    "low <= open": (bars_df["low"] <= bars_df["open"]).all(),

    "low <= close": (bars_df["low"] <= bars_df["close"]).all(),

    "volume is non-negative": (bars_df["volume"] >= 0).all(),

    "trade count is non-negative": (bars_df["trade_count"] >= 0).all(),

    "vwap is within high/low": (
        (bars_df["vwap"] >= bars_df["low"])
        & (bars_df["vwap"] <= bars_df["high"])
    ).all(),
}

for check_name, passed in checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"{status}: {check_name}")

PASS: no missing values
PASS: unique symbol/timestamp
PASS: high >= open
PASS: high >= close
PASS: low <= open
PASS: low <= close
PASS: volume is non-negative
PASS: trade count is non-negative
PASS: vwap is within high/low


In [10]:
invalid_ohlc = bars_df[
    (bars_df["high"] < bars_df["open"])
    | (bars_df["high"] < bars_df["close"])
    | (bars_df["low"] > bars_df["open"])
    | (bars_df["low"] > bars_df["close"])
]

print("Invalid OHLC rows:", len(invalid_ohlc))
display(invalid_ohlc)

Invalid OHLC rows: 0


,symbol,timestamp,open,high,low,close,volume,trade_count,vwap


In [11]:
# print(type(response))
# print(response.data.keys())

# bars = response.data[symbol]

# print("record count:", len(bars))
# print("record type:", type(bars[0]))
# print("record:", bars[0])
# print("serializable record:", bars[0].model_dump(mode="json"))

In [12]:
# # multi symbol request - single symbol is similar
# multisymbol_request_params = StockLatestQuoteRequest(symbol_or_symbols=["SPY", "GLD", "TLT"])

# latest_multisymbol_quotes = client.get_stock_latest_quote(multisymbol_request_params)

# gld_latest_ask_price = latest_multisymbol_quotes["GLD"].ask_price

In [13]:
# gld_latest_ask_price

In [14]:
# latest_multisymbol_quotes["TLT"]

In [15]:
trading_client = TradingClient(settings.alpaca_api_key.get_secret_value(), settings.alpaca_secret_key.get_secret_value(), paper=True)
clock = trading_client.get_clock()

print("Open:", clock.is_open)
print("Next open:", clock.next_open)
print("Next close:", clock.next_close)
# print(trading_client.get_all_assets('NASDAQ'))

Open: True
Next open: 2026-10-26 09:30:00-04:00
Next close: 2026-10-23 16:00:00-04:00


In [16]:
from alpaca.trading.requests import GetCalendarRequest
from datetime import date

calendar_request = GetCalendarRequest(
    start=date(2026, 10, 1),
    end=date(2026, 10, 16),
)

calendar = trading_client.get_calendar(calendar_request)

print("Session count:", len(calendar))

for session in calendar:
    print(session.model_dump(mode="json"))

Session count: 12
{'date': '2026-10-01', 'open': '2026-10-01T09:30:00', 'close': '2026-10-01T16:00:00'}
{'date': '2026-10-02', 'open': '2026-10-02T09:30:00', 'close': '2026-10-02T16:00:00'}
{'date': '2026-10-05', 'open': '2026-10-05T09:30:00', 'close': '2026-10-05T16:00:00'}
{'date': '2026-10-06', 'open': '2026-10-06T09:30:00', 'close': '2026-10-06T16:00:00'}
{'date': '2026-10-07', 'open': '2026-10-07T09:30:00', 'close': '2026-10-07T16:00:00'}
{'date': '2026-10-08', 'open': '2026-10-08T09:30:00', 'close': '2026-10-08T16:00:00'}
{'date': '2026-10-09', 'open': '2026-10-09T09:30:00', 'close': '2026-10-09T16:00:00'}
{'date': '2026-10-12', 'open': '2026-10-12T09:30:00', 'close': '2026-10-12T16:00:00'}
{'date': '2026-10-13', 'open': '2026-10-13T09:30:00', 'close': '2026-10-13T16:00:00'}
{'date': '2026-10-14', 'open': '2026-10-14T09:30:00', 'close': '2026-10-14T16:00:00'}
{'date': '2026-10-15', 'open': '2026-10-15T09:30:00', 'close': '2026-10-15T16:00:00'}
{'date': '2026-10-16', 'open': '2026

In [17]:
from datetime import date, datetime

import pandas as pd
from alpaca.data.enums import DataFeed
from alpaca.data.historical.corporate_actions import CorporateActionsClient
from alpaca.data.requests import (
    CorporateActionsRequest,
    StockQuotesRequest,
    StockTradesRequest,
)
from alpaca.trading.client import TradingClient
from alpaca.trading.requests import GetCalendarRequest

from equity_data_platform.clients import create_historical_client
from equity_data_platform.config import get_settings

In [18]:
settings = get_settings()

api_key = settings.alpaca_api_key.get_secret_value()
secret_key = settings.alpaca_secret_key.get_secret_value()

symbols = list(settings.symbols)
feed = DataFeed(settings.feed)

historical_client = create_historical_client(settings)

corporate_actions_client = CorporateActionsClient(
    api_key=api_key,
    secret_key=secret_key,
)

trading_client = TradingClient(
    api_key=api_key,
    secret_key=secret_key,
    paper=True,
)

print("Symbols:", symbols)
print("Feed:", feed.value)

Symbols: ['AAPL', 'MSFT', 'NVDA', 'AMZN', 'META']
Feed: iex


In [19]:
def inspect_dataframe(name: str, df: pd.DataFrame) -> None:
    print(f"{name} shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if df.empty:
        print("No records returned.")
        return

    print("\nData types:")
    print(df.dtypes)

    print("\nSample:")
    display(df.head())

In [20]:
test_symbol = symbols[0]

test_start = datetime(2026, 10, 16, 14, 30)
test_end = datetime(2026, 10, 16, 14, 35)

In [21]:
trades_request = StockTradesRequest(
    symbol_or_symbols=test_symbol,
    start=test_start,
    end=test_end,
    feed=feed,
    limit=1_000,
)

trades_response = historical_client.get_stock_trades(
    trades_request
)

trades_df = trades_response.df.reset_index()

inspect_dataframe("Trades", trades_df)

Trades shape: (422, 8)
Columns: ['symbol', 'timestamp', 'exchange', 'price', 'size', 'id', 'conditions', 'tape']

Data types:
symbol                        str
timestamp     datetime64[us, UTC]
exchange                      str
price                     float64
size                      float64
id                          int64
conditions                 object
tape                          str
dtype: object

Sample:


,symbol,timestamp,exchange,price,size,id,conditions,tape
0,AAPL,2026-10-16 14:30:04.247065+00:00,V,344.275,20.0,7923,"[@, I]",C
1,AAPL,2026-10-16 14:30:04.250956+00:00,V,344.275,20.0,7924,"[@, I]",C
2,AAPL,2026-10-16 14:30:04.258905+00:00,V,344.275,62.0,7925,[@],C
3,AAPL,2026-10-16 14:30:04.335658+00:00,V,344.250,37.0,7926,"[@, I]",C
4,AAPL,2026-10-16 14:30:04.335732+00:00,V,344.250,3.0,7927,"[@, I]",C


In [26]:
trade_records = trades_response.data.get(test_symbol, [])

if trade_records:
    print(trade_records[0].model_dump(mode="json"))
else:
    print("No trades returned for", test_symbol)

{'symbol': 'AAPL', 'timestamp': '2026-10-16T14:30:04.247065Z', 'exchange': 'V', 'price': 344.275, 'size': 20.0, 'id': 7923, 'conditions': ['@', 'I'], 'tape': 'C'}


In [27]:
quotes_request = StockQuotesRequest(
    symbol_or_symbols=test_symbol,
    start=test_start,
    end=test_end,
    feed=feed,
    limit=1_000,
)

quotes_response = historical_client.get_stock_quotes(
    quotes_request
)

quotes_df = quotes_response.df.reset_index()

inspect_dataframe("Quotes", quotes_df)

Quotes shape: (1000, 10)
Columns: ['symbol', 'timestamp', 'bid_price', 'bid_size', 'bid_exchange', 'ask_price', 'ask_size', 'ask_exchange', 'conditions', 'tape']

Data types:
symbol                          str
timestamp       datetime64[us, UTC]
bid_price                   float64
bid_size                    float64
bid_exchange                    str
ask_price                   float64
ask_size                    float64
ask_exchange                    str
conditions                   object
tape                            str
dtype: object

Sample:


,symbol,timestamp,bid_price,bid_size,bid_exchange,ask_price,ask_size,ask_exchange,conditions,tape
0,AAPL,2026-10-16 14:30:00.098224+00:00,343.75,80.0,V,344.38,40.0,V,[R],C
1,AAPL,2026-10-16 14:30:00.098970+00:00,344.26,40.0,V,344.38,40.0,V,[R],C
2,AAPL,2026-10-16 14:30:00.121606+00:00,344.26,40.0,V,344.38,40.0,V,[R],C
3,AAPL,2026-10-16 14:30:00.170124+00:00,344.25,40.0,V,344.38,40.0,V,[R],C
4,AAPL,2026-10-16 14:30:00.170430+00:00,344.25,40.0,V,344.38,40.0,V,[R],C


In [28]:
quote_records = quotes_response.data.get(test_symbol, [])

if quote_records:
    print(quote_records[0].model_dump(mode="json"))
else:
    print("No quotes returned for", test_symbol)

{'symbol': 'AAPL', 'timestamp': '2026-10-16T14:30:00.098224Z', 'bid_price': 343.75, 'bid_size': 80.0, 'bid_exchange': 'V', 'ask_price': 344.38, 'ask_size': 40.0, 'ask_exchange': 'V', 'conditions': ['R'], 'tape': 'C'}


In [29]:
corporate_actions_request = CorporateActionsRequest(
    symbols=symbols,
    start=date(2020, 1, 1),
    end=date.today(),
    limit=1_000,
)

corporate_actions_response = (
    corporate_actions_client.get_corporate_actions(
        corporate_actions_request
    )
)

corporate_actions_df = (
    corporate_actions_response.df.reset_index()
)

inspect_dataframe(
    "Corporate actions",
    corporate_actions_df,
)

Corporate actions shape: (101, 23)
Columns: ['corporate_action_type', 'id', 'symbol', 'cusip', 'rate', 'special', 'foreign', 'process_date', 'ex_date', 'record_date', 'payable_date', 'acquirer_symbol', 'acquirer_cusip', 'acquiree_symbol', 'acquiree_cusip', 'effective_date', 'new_rate', 'old_rate', 'due_bill_redemption_date', 'old_symbol', 'old_cusip', 'new_symbol', 'new_cusip']

Data types:
corporate_action_type           str
id                           object
symbol                          str
cusip                           str
rate                        float64
special                      object
foreign                      object
process_date                 object
ex_date                      object
record_date                  object
payable_date                 object
acquirer_symbol                 str
acquirer_cusip                  str
acquiree_symbol                 str
acquiree_cusip                  str
effective_date               object
new_rate                    fl

,corporate_action_type,id,symbol,cusip,rate,special,foreign,process_date,ex_date,record_date,...,acquiree_symbol,acquiree_cusip,effective_date,new_rate,old_rate,due_bill_redemption_date,old_symbol,old_cusip,new_symbol,new_cusip
0,cash_dividends,23407e01-4eef-4c1e-a2dd-f585c7eabcd0,AAPL,,0.770,False,False,2020-02-07,2020-02-07,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,cash_dividends,235946d4-2605-4e5c-9699-9bfd00452ec0,AAPL,037833100,0.820,False,False,2020-05-14,2020-05-08,2020-05-11,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,cash_dividends,d09386ae-0c2b-4280-8aa0-295e545354b1,AAPL,037833100,0.820,False,False,2020-08-13,2020-08-07,2020-08-10,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,cash_dividends,43032b74-6743-4fef-9a10-de51c464a480,AAPL,037833100,0.205,False,False,2020-11-12,2020-11-06,2020-11-09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,cash_dividends,65570a16-85fa-4c01-b448-381a789c9248,AAPL,037833100,0.205,False,False,2021-02-11,2021-02-05,2021-02-08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [30]:
print("Response groups:")
print(corporate_actions_response.data.keys())

Response groups:
dict_keys(['cash_dividends', 'cash_mergers', 'forward_splits', 'name_changes'])


In [31]:
asset_records = []

for symbol in symbols:
    asset = trading_client.get_asset(symbol)
    asset_records.append(asset.model_dump(mode="json"))

assets_df = pd.DataFrame(asset_records)

inspect_dataframe("Assets", assets_df)

Assets shape: (5, 16)
Columns: ['id', 'asset_class', 'exchange', 'symbol', 'name', 'status', 'tradable', 'marginable', 'shortable', 'easy_to_borrow', 'fractionable', 'min_order_size', 'min_trade_increment', 'price_increment', 'maintenance_margin_requirement', 'attributes']

Data types:
id                                    str
asset_class                           str
exchange                              str
symbol                                str
name                                  str
status                                str
tradable                             bool
marginable                           bool
shortable                            bool
easy_to_borrow                       bool
fractionable                         bool
min_order_size                     object
min_trade_increment                object
price_increment                    object
maintenance_margin_requirement    float64
attributes                         object
dtype: object

Sample:


,id,asset_class,exchange,symbol,name,status,tradable,marginable,shortable,easy_to_borrow,fractionable,min_order_size,min_trade_increment,price_increment,maintenance_margin_requirement,attributes
0,b0b6dd9d-8b9b-48a9-ba46-b9d54906e415,us_equity,NASDAQ,AAPL,Apple Inc. Common Stock,active,True,True,True,True,True,None,None,None,30.0,"[fractional_eh_enabled, has_options, overnight..."
1,b6d1aa75-5c9c-4353-a305-9e2caa1925ab,us_equity,NASDAQ,MSFT,Microsoft Corporation Common Stock,active,True,True,True,True,True,None,None,None,30.0,"[fractional_eh_enabled, has_options, overnight..."
2,4ce9353c-66d1-46c2-898f-fce867ab0247,us_equity,NASDAQ,NVDA,NVIDIA Corporation Common Stock,active,True,True,True,True,True,None,None,None,30.0,"[fractional_eh_enabled, has_options, overnight..."
3,f801f835-bfe6-4a9d-a6b1-ccbb84bfd75f,us_equity,NASDAQ,AMZN,"Amazon.com, Inc. Common Stock",active,True,True,True,True,True,None,None,None,30.0,"[fractional_eh_enabled, has_options, overnight..."
4,fc6a5dcd-4a70-4b8d-b64f-d83a6dae9ba4,us_equity,NASDAQ,META,"Meta Platforms, Inc. Class A Common Stock",active,True,True,True,True,True,None,None,None,30.0,"[fractional_eh_enabled, has_options, overnight..."


In [22]:
# wss_client = StockDataStream(api_key, secret_key)

# async def quote_data_handler(quote):
#     print(quote)

# wss_client.subscribe_quotes(
#     quote_data_handler,
#     "SPY", "QQQ", "AAPL", "MSFT", "NVDA"
# )

# stream_thread = threading.Thread(
#     target=wss_client.run,
#     daemon=True,
# )

In [23]:
# stream_thread.start()

In [24]:
# wss_client.stop()
# stream_thread.join(timeout=5)

In [25]:
# if stream_thread.is_alive():
#     print("The stream thread did not stop within 5 seconds")
# else:
#     print("Stream stopped successfully")